# 🧠 Brain Tumor Segmentation (BraTS 2023) — Notebook Huấn luyện 2 Mô hình

Notebook này hướng dẫn quy trình huấn luyện và đánh giá hai mô hình Deep Learning phân đoạn khối u não 3D:
1. **3D U-Net** (Convolutional Encoder-Decoder Baseline)
2. **Swin UNETR** (Swin Transformer 3D Encoder + UNet Decoder)

Sử dụng thư viện **PyTorch** & **MONAI**, hỗ trợ chạy trên **Google Colab** (GPU T4/P100/A100) hoặc máy cá nhân.


## 1. Cài đặt các thư viện cần thiết


## 2. Thống nhất Môi trường & Khai báo Thư viện


In [2]:
import os
import sys
import time
import math
import json
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import monai
import monai.transforms as mt
from monai.networks.nets import UNet, SwinUNETR
from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference

# Kiểm tra GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f" GPU Device: {torch.cuda.get_device_name(0)}")

 CUDA Available: True
 GPU Device: Tesla T4


## 3. Cấu hình Đường dẫn Dữ liệu BraTS 2023


In [3]:
# Đường dẫn dataset
# Nếu chạy trên Colab, mount Google Drive trước khi tìm dataset.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

# Ưu tiên đường dẫn trong biến môi trường, sau đó thử các vị trí Google Drive phổ biến.
DATA_ROOT_CANDIDATES = [
    os.environ.get("BRATS_DATA_ROOT"),
    os.environ.get("DATA_ROOT"),
    "Datasets/brats2023-gli-dataset/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData",
    "/content/drive/MyDrive/BraTS2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData",
    "/content/drive/MyDrive/SIC_Capstone_2026/data/BraTS2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData",
    "/content/drive/MyDrive/SIC_Capstone_2026/data/BraTS2023/brats2023-gli-dataset/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData",
]

DATA_ROOT = None
for candidate in DATA_ROOT_CANDIDATES:
    if not candidate:
        continue
    path = Path(candidate)
    case_dirs = sorted([d.name for d in path.iterdir() if d.is_dir()]) if path.exists() else []
    if case_dirs:
        DATA_ROOT = path
        all_cases = case_dirs
        break

if DATA_ROOT is None:
    DATA_ROOT = Path(DATA_ROOT_CANDIDATES[2])
    all_cases = []
    print("Không tìm thấy dataset. Hãy mount Google Drive và set đúng path, ví dụ:")
    print('os.environ["BRATS_DATA_ROOT"] = "/content/drive/MyDrive/.../ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"')

print(f" Data Root Path: {DATA_ROOT.resolve()}")
print(f" Tổng số cases tìm thấy: {len(all_cases)}")
if len(all_cases) > 0:
    print(f"Sample cases: {all_cases[:5]}")

Mounted at /content/drive
 Data Root Path: /content/drive/MyDrive/BraTS2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData
 Tổng số cases tìm thấy: 1251
Sample cases: ['BraTS-GLI-00000-000', 'BraTS-GLI-00002-000', 'BraTS-GLI-00003-000', 'BraTS-GLI-00005-000', 'BraTS-GLI-00006-000']


## 3.5 Data Understanding — Tìm hiểu Dữ liệu

Trước khi huấn luyện, ta khảo sát cấu trúc dữ liệu BraTS 2023 GLI:
- **4 modalities**: `t1n`, `t1c`, `t2w`, `t2f` + 1 mask `seg` cho mỗi case.
- **Nhãn (labels)**: `1`=NCR (hoại tử), `2`=ED (phù nề), `3`/`4`=ET (u tăng cường), `0`=nền.
- Kiểm tra: kích thước ảnh, voxel spacing, tính toàn vẹn file, và phân bố train/val.


In [ ]:
# ── 1. Structure inspection: load one case, print shape / spacing / dtype ──
if all_cases:
    _case = all_cases[0]
    _case_dir = DATA_ROOT / _case
    _modalities = ["t1n", "t1c", "t2w", "t2f"]

    print(f"Case: {_case}")
    for mod in _modalities:
        _nii = nib.load(_case_dir / f"{_case}-{mod}.nii.gz")
        _arr = _nii.get_fdata()
        _spacing = np.abs(np.diag(_nii.affine)[:3])
        print(f"  {mod}: shape={_arr.shape}, dtype={_arr.dtype}, "
              f"spacing={_spacing.round(3)}, range=[{_arr.min():.1f}, {_arr.max():.1f}]")

    # ── 2. Label inventory: unique label values in seg ──
    _seg = nib.load(_case_dir / f"{_case}-seg.nii.gz").get_fdata().astype(np.int32)
    print(f"\nLabel unique values (raw): {np.unique(_seg)}")
    _label_names = {0: "Background", 1: "NCR", 2: "ED", 3: "ET(3)", 4: "ET(4)"}
    for v in np.unique(_seg):
        pct = 100.0 * (_seg == v).sum() / _seg.size
        print(f"  Label {v} ({_label_names.get(v,'?')}): {pct:.2f}% of voxels")

    # ── 3. Integrity check: scan all cases for missing files ──
    MODALITIES_ALL = ["t1n", "t1c", "t2w", "t2f", "seg"]
    missing = {}
    for case_id in all_cases:
        case_dir = DATA_ROOT / case_id
        absent = [m for m in MODALITIES_ALL
                  if not (case_dir / f"{case_id}-{m}.nii.gz").exists()]
        if absent:
            missing[case_id] = absent
    print(f"\nIntegrity check: {len(all_cases)} cases scanned")
    print(f"  Cases with missing files: {len(missing)}")
    if missing:
        for cid, mods in list(missing.items())[:5]:
            print(f"    {cid}: missing {mods}")

    # ── 4. Split summary ──
    np.random.seed(42)
    _shuffled = all_cases.copy()
    np.random.shuffle(_shuffled)
    _n_val = max(1, int(len(_shuffled) * 0.15))
    print(f"\nSplit summary (seed=42): Train={len(_shuffled)-_n_val}, Val={_n_val}, "
          f"Total={len(all_cases)}")
else:
    print("⚠️ No cases found — mount Google Drive and set DATA_ROOT first.")

## 3.6 EDA — Phân tích Khám phá Dữ liệu (Exploratory Data Analysis)

Trực quan hóa và thống kê để hiểu đặc điểm dữ liệu trước khi huấn luyện:
- **EDA 1**: Hiển thị 4 modalities cùng một lát cắt + overlay mask phân đoạn (chọn lát cắt nhiều u nhất).
- **EDA 2**: Phân bố cường độ (intensity) từng modality + thống kê tỉ lệ xuất hiện và thể tích của các vùng u (NCR/ED/ET) trên tập con ngẫu nhiên.


In [ ]:
# ── EDA 1: Multi-modal slice visualization + segmentation overlay ──
if all_cases:
    _case = all_cases[0]
    _cdir = DATA_ROOT / _case
    _mods = ["t1n", "t1c", "t2w", "t2f"]

    # Load 4 modalities + segmentation mask
    _vols = {m: nib.load(_cdir / f"{_case}-{m}.nii.gz").get_fdata() for m in _mods}
    _seg = nib.load(_cdir / f"{_case}-seg.nii.gz").get_fdata().astype(np.int32)

    # Chọn lát cắt axial có nhiều voxel u nhất để nhìn rõ khối u
    _tumor_per_slice = (_seg > 0).sum(axis=(0, 1))
    _z = int(np.argmax(_tumor_per_slice)) if _tumor_per_slice.max() > 0 else _seg.shape[2] // 2

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    # (a) 4 modalities tại cùng một lát cắt
    for i, m in enumerate(_mods):
        axes[i].imshow(np.rot90(_vols[m][:, :, _z]), cmap="gray")
        axes[i].set_title(m, fontweight="bold")
        axes[i].axis("off")

    # (b) Overlay mask phân đoạn (remap 4->3) lên FLAIR (t2f)
    _seg_vis = _seg.copy(); _seg_vis[_seg_vis == 4] = 3
    _flair = np.rot90(_vols["t2f"][:, :, _z])
    _mask = np.rot90(_seg_vis[:, :, _z])
    axes[4].imshow(_flair, cmap="gray")
    axes[4].imshow(np.ma.masked_where(_mask == 0, _mask), cmap="jet", alpha=0.6, vmin=1, vmax=3)
    axes[4].set_title("t2f + Seg overlay\n(1=NCR, 2=ED, 3=ET)", fontweight="bold")
    axes[4].axis("off")

    fig.suptitle(f"EDA 1 — {_case} | axial slice z={_z}", fontweight="bold")
    plt.tight_layout(); plt.show()
else:
    print("⚠️ No cases found — mount Google Drive and set DATA_ROOT first.")

In [ ]:
# ── EDA 2: Intensity distributions + tumor class statistics over a subset ──
import pandas as pd

if all_cases:
    # (a) Raw intensity histograms per modality (brain voxels only) for one case
    _case = all_cases[0]
    _cdir = DATA_ROOT / _case
    fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))
    for i, mod in enumerate(["t1n", "t1c", "t2w", "t2f"]):
        vol = nib.load(_cdir / f"{_case}-{mod}.nii.gz").get_fdata()
        vals = vol[vol > 0]  # bỏ background
        axes[i].hist(vals, bins=80, color="steelblue", alpha=0.8)
        axes[i].set_title(f"{mod} (raw)", fontweight="bold")
        axes[i].set_xlabel("Intensity"); axes[i].set_yscale("log")
    fig.suptitle(f"Raw Intensity Distributions — {_case}", fontweight="bold")
    plt.tight_layout(); plt.show()

    # (b) Tumor class statistics over a random subset (dùng seg đã remap {0,1,2,3})
    rng = np.random.default_rng(42)
    N_SUBSET = min(100, len(all_cases))
    subset = rng.choice(all_cases, size=N_SUBSET, replace=False)
    LABEL_MAP = {0: 0, 1: 1, 2: 2, 3: 3, 4: 3}
    class_names = {1: "NCR", 2: "ED", 3: "ET"}

    rows = []
    for cid in subset:
        seg = nib.load(DATA_ROOT / cid / f"{cid}-seg.nii.gz").get_fdata().astype(np.int64)
        remapped = np.zeros_like(seg)
        for s, d in LABEL_MAP.items():
            remapped[seg == s] = d
        brain_vox = int((remapped >= 0).sum())  # toàn ảnh; xem tỉ lệ u/ảnh
        tumor_vox = int((remapped > 0).sum())
        row = {"case": cid, "tumor_ratio": tumor_vox / brain_vox}
        for lab, name in class_names.items():
            row[f"vox_{name}"] = int((remapped == lab).sum())
        rows.append(row)

    df = pd.DataFrame(rows)
    print(f"Tumor class statistics over {N_SUBSET} random cases:\n")
    for lab, name in class_names.items():
        col = f"vox_{name}"
        present = (df[col] > 0).mean() * 100
        print(f"  {name:<4}: present in {present:5.1f}% of cases | "
              f"mean voxels = {df[col].mean():,.0f} | median = {df[col].median():,.0f}")
    print(f"\n  Tumor-to-image volume ratio: mean = {df['tumor_ratio'].mean()*100:.3f}%, "
          f"max = {df['tumor_ratio'].max()*100:.3f}%")

    # (c) Volume distribution box plots per tumor class
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.boxplot([df["vox_NCR"], df["vox_ED"], df["vox_ET"]],
               labels=["NCR", "ED", "ET"], showfliers=True)
    ax.set_title("Tumor Sub-region Volume Distribution (voxels)", fontweight="bold")
    ax.set_ylabel("Voxel count"); ax.set_yscale("log")
    ax.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout(); plt.show()

## 4. Định nghĩa BraTS Dataset Loader


In [ ]:
class BraTSDataset3D(Dataset):
    """Dataset custom đọc NIfTI MRI 3D (4 channels: T1n, T1c, T2w, T2f + Mask).

    Tiền xử lý (preprocessing) đã tích hợp:
      1. Percentile clipping [0.5, 99.5] trên vùng não để loại bỏ nhiễu/outlier.
      2. Z-score normalization CHỈ trên vùng não (bỏ background) để thống kê không bị lệch.
      3. Remap nhãn (label): 4 -> 3 để nhãn liên tục {0,1,2,3} khớp out_channels=4.
    """
    MODALITIES = ["t1n", "t1c", "t2w", "t2f"]
    # BraTS raw labels -> nhãn training liên tục (xử lý cả trường hợp legacy ET=4)
    LABEL_MAP = {0: 0, 1: 1, 2: 2, 3: 3, 4: 3}

    def __init__(self, data_root: Path, case_list: List[str], transforms=None):
        self.data_root = Path(data_root)
        self.case_list = case_list
        self.transforms = transforms

    def __len__(self):
        return len(self.case_list)

    @staticmethod
    def _clip_and_normalize(volume: np.ndarray) -> np.ndarray:
        """Clip percentile + z-score trên vùng não (voxel > 0)."""
        brain = volume > 0  # background của MRI não = 0
        if brain.sum() > 0:
            vals = volume[brain]
            lo, hi = np.percentile(vals, [0.5, 99.5])
            volume = np.clip(volume, lo, hi)
            mean = volume[brain].mean()
            std = volume[brain].std()
            if std > 0:
                volume = (volume - mean) / std
            volume[~brain] = 0.0  # giữ background = 0
        return volume.astype(np.float32)

    def _remap_label(self, label: np.ndarray) -> np.ndarray:
        """Ánh xạ nhãn BraTS về {0,1,2,3}."""
        remapped = np.zeros_like(label, dtype=np.float32)
        for src, dst in self.LABEL_MAP.items():
            remapped[label == src] = dst
        return remapped

    def __getitem__(self, idx):
        case_id = self.case_list[idx]
        case_dir = self.data_root / case_id

        # Load 4 modalities
        mods = []
        for mod in self.MODALITIES:
            file_path = case_dir / f"{case_id}-{mod}.nii.gz"
            nii = nib.load(file_path)
            mods.append(nii.get_fdata())

        image = np.stack(mods, axis=0).astype(np.float32)  # (4, H, W, D)

        # Tiền xử lý cường độ: clip percentile + z-score theo vùng não (per channel)
        for c in range(4):
            image[c] = self._clip_and_normalize(image[c])

        # Load segmentation mask if available
        seg_file = case_dir / f"{case_id}-seg.nii.gz"
        if seg_file.exists():
            seg_nii = nib.load(seg_file)
            raw_label = seg_nii.get_fdata().astype(np.int64)
            label = self._remap_label(raw_label)          # {0,1,2,3}
            label = np.expand_dims(label, axis=0)          # (1, H, W, D)
        else:
            label = np.zeros((1,) + image.shape[1:], dtype=np.float32)

        data_dict = {"image": torch.from_numpy(image), "label": torch.from_numpy(label), "case_id": case_id}

        if self.transforms:
            data_dict = self.transforms(data_dict)

        return data_dict

# Train / Val Split
np.random.seed(42)
np.random.shuffle(all_cases)
n_val = max(1, int(len(all_cases) * 0.15))
train_cases = all_cases[:-n_val]
val_cases = all_cases[-n_val:]

print(f"✂️ Data Split: Train = {len(train_cases)} cases | Validation = {len(val_cases)} cases")

## 5. MONAI Data Augmentation & DataLoader Setup


In [5]:
# Transforms cho Training (Patch size 128x128x128)
train_transforms = mt.Compose([
    mt.RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=(128, 128, 128),
        pos=1.0, neg=1.0,
        num_samples=2,
        image_key="image"
    ),
    mt.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    mt.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    mt.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
    mt.EnsureTyped(keys=["image", "label"]),
])

val_transforms = mt.Compose([
    mt.EnsureTyped(keys=["image", "label"]),
])

# Tạo Datasets
train_dataset = BraTSDataset3D(DATA_ROOT, train_cases, transforms=train_transforms)
val_dataset = BraTSDataset3D(DATA_ROOT, val_cases, transforms=val_transforms)

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=0)

print("✅ DataLoaders initialized successfully!")

✅ DataLoaders initialized successfully!


### 5.1 Sanity Check — Kiểm tra DataLoader trước khi Huấn luyện

Xác minh pipeline dữ liệu hoạt động đúng: kích thước batch/patch, khoảng giá trị nhãn `{0,1,2,3}`, cường độ đã z-score, và trực quan hóa một patch đã được augment kèm mask.


In [ ]:
# ── DataLoader sanity-check: batch shapes, label range, augmented patch viz ──
_check_batch = next(iter(train_loader))
if isinstance(_check_batch, list):          # RandCropByPosNegLabeld trả về list nhiều patch
    _check_batch = _check_batch[0]

_img = _check_batch["image"]
_lbl = _check_batch["label"]

print("Batch image shape :", tuple(_img.shape), "| dtype:", _img.dtype)
print("Batch label shape :", tuple(_lbl.shape), "| dtype:", _lbl.dtype)
print(f"Image value range : [{_img.min():.3f}, {_img.max():.3f}]  (đã z-score theo vùng não)")

_uniq = torch.unique(_lbl)
print("Label unique values:", _uniq.tolist())

# Kiểm tra bất biến (invariants) của pipeline
assert _img.shape[1] == 4, "Ảnh phải có 4 channel (t1n, t1c, t2w, t2f)"
assert tuple(_img.shape[-3:]) == (128, 128, 128), "Patch phải là 128x128x128"
assert _lbl.min() >= 0 and _lbl.max() <= 3, "Nhãn phải nằm trong {0,1,2,3}"
print("✅ Sanity-check passed: shapes & label range hợp lệ.")

# Trực quan hóa 1 patch đã augment: 4 modalities + mask tại lát cắt giữa
_patch = _img[0].cpu().numpy()          # (4, 128, 128, 128)
_pmask = _lbl[0, 0].cpu().numpy()       # (128, 128, 128)
_zc = _patch.shape[-1] // 2

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, m in enumerate(["t1n", "t1c", "t2w", "t2f"]):
    axes[i].imshow(np.rot90(_patch[i, :, :, _zc]), cmap="gray")
    axes[i].set_title(f"{m} (patch)", fontweight="bold")
    axes[i].axis("off")

_mslice = np.rot90(_pmask[:, :, _zc])
axes[4].imshow(np.rot90(_patch[0, :, :, _zc]), cmap="gray")
axes[4].imshow(np.ma.masked_where(_mslice == 0, _mslice), cmap="jet", alpha=0.6, vmin=1, vmax=3)
axes[4].set_title("Augmented patch + mask", fontweight="bold")
axes[4].axis("off")

fig.suptitle("DataLoader Sanity Check — augmented training patch", fontweight="bold")
plt.tight_layout(); plt.show()

## 6. Huấn luyện Mô hình 1: 3D U-Net Baseline


In [ ]:
# 1. Khởi tạo kiến trúc 3D U-Net
unet_model = UNet(
    spatial_dims=3,
    in_channels=4,
    out_channels=4,  # Background + 3 Tumor classes
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    dropout=0.2,
    norm="instance"
).to(device)

loss_function = DiceCELoss(to_onehot_y=True, softmax=True, include_background=True)
optimizer_unet = torch.optim.AdamW(unet_model.parameters(), lr=2e-4, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

num_epochs = 20  # Có thể tăng lên 50-100 khi train trên GPU
val_interval = 2
best_unet_dice = -1.0
unet_history = {"train_loss": [], "val_dice": []}

print("🚀 Bắt đầu huấn luyện 3D U-Net...")

for epoch in range(1, num_epochs + 1):
    unet_model.train()
    epoch_loss = 0.0
    start_t = time.time()

    for batch in train_loader:
        # Handling list of patches from RandCropByPosNegLabeld
        if isinstance(batch, list):
            batch = batch[0]

        imgs = batch["image"].to(device)
        lbls = batch["label"].to(device)

        optimizer_unet.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            outputs = unet_model(imgs)
            loss = loss_function(outputs, lbls)

        scaler.scale(loss).backward()
        scaler.step(optimizer_unet)
        scaler.update()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / max(1, len(train_loader))
    unet_history["train_loss"].append(avg_loss)
    print(f"Epoch [{epoch:02d}/{num_epochs:02d}] Train Loss: {avg_loss:.4f} | Time: {time.time()-start_t:.1f}s")

    # Validation
    if epoch % val_interval == 0:
        unet_model.eval()
        dice_scores = []
        with torch.no_grad():
            for val_batch in val_loader:
                v_imgs = val_batch["image"].to(device)
                v_lbls = val_batch["label"].to(device)

                # Sliding window inference
                val_outputs = sliding_window_inference(v_imgs, roi_size=(128, 128, 128), sw_batch_size=4, predictor=unet_model)
                val_preds = (torch.softmax(val_outputs, dim=1) > 0.5).float()

                # Calculate simple Dice
                intersection = torch.sum(val_preds * v_lbls)
                dice = (2.0 * intersection + 1e-5) / (torch.sum(val_preds) + torch.sum(v_lbls) + 1e-5)
                dice_scores.append(dice.item())

        mean_dice = float(np.mean(dice_scores)) if len(dice_scores) > 0 else 0.0
        unet_history["val_dice"].append(mean_dice)
        print(f"   ⭐ Validation Dice: {mean_dice:.4f}")

        if mean_dice > best_unet_dice:
            best_unet_dice = mean_dice
            os.makedirs("checkpoints", exist_ok=True)
            torch.save(unet_model.state_dict(), "checkpoints/unet_brats_best.pth")
            print(f"   💾 Saved Best 3D U-Net Checkpoint! Dice = {best_unet_dice:.4f}")

print("✅ Hoàn thành huấn luyện 3D U-Net!")

/tmp/ipykernel_1018/422190270.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


🚀 Bắt đầu huấn luyện 3D U-Net...


/tmp/ipykernel_1018/422190270.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):


## 7. Huấn luyện Mô hình 2: Swin UNETR (3D Transformer)


In [ ]:
# 1. Khởi tạo kiến trúc Swin UNETR
swin_model = SwinUNETR(
    img_size=(128, 128, 128),
    in_channels=4,
    out_channels=4,
    feature_size=48,
    use_checkpoint=True,
    spatial_dims=3
).to(device)

optimizer_swin = torch.optim.AdamW(swin_model.parameters(), lr=2e-4, weight_decay=1e-4)
best_swin_dice = -1.0
swin_history = {"train_loss": [], "val_dice": []}

print("🚀 Bắt đầu huấn luyện Swin UNETR...")

for epoch in range(1, num_epochs + 1):
    swin_model.train()
    epoch_loss = 0.0
    start_t = time.time()

    for batch in train_loader:
        if isinstance(batch, list):
            batch = batch[0]

        imgs = batch["image"].to(device)
        lbls = batch["label"].to(device)

        optimizer_swin.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            outputs = swin_model(imgs)
            loss = loss_function(outputs, lbls)

        scaler.scale(loss).backward()
        scaler.step(optimizer_swin)
        scaler.update()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / max(1, len(train_loader))
    swin_history["train_loss"].append(avg_loss)
    print(f"Epoch [{epoch:02d}/{num_epochs:02d}] Train Loss: {avg_loss:.4f} | Time: {time.time()-start_t:.1f}s")

    # Validation
    if epoch % val_interval == 0:
        swin_model.eval()
        dice_scores = []
        with torch.no_grad():
            for val_batch in val_loader:
                v_imgs = val_batch["image"].to(device)
                v_lbls = val_batch["label"].to(device)

                val_outputs = sliding_window_inference(v_imgs, roi_size=(128, 128, 128), sw_batch_size=2, predictor=swin_model)
                val_preds = (torch.softmax(val_outputs, dim=1) > 0.5).float()

                intersection = torch.sum(val_preds * v_lbls)
                dice = (2.0 * intersection + 1e-5) / (torch.sum(val_preds) + torch.sum(v_lbls) + 1e-5)
                dice_scores.append(dice.item())

        mean_dice = float(np.mean(dice_scores)) if len(dice_scores) > 0 else 0.0
        swin_history["val_dice"].append(mean_dice)
        print(f"   ⭐ Validation Dice: {mean_dice:.4f}")

        if mean_dice > best_swin_dice:
            best_swin_dice = mean_dice
            os.makedirs("checkpoints", exist_ok=True)
            torch.save(swin_model.state_dict(), "checkpoints/swin_unetr_brats_best.pth")
            print(f"   💾 Saved Best Swin UNETR Checkpoint! Dice = {best_swin_dice:.4f}")

print("✅ Hoàn thành huấn luyện Swin UNETR!")

## 8. Đánh giá & So sánh Hiệu năng hai Mô hình


In [ ]:
# Draw Training Loss & Validation Dice Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Loss
axes[0].plot(unet_history["train_loss"], label="3D U-Net", color="blue", linewidth=2)
axes[0].plot(swin_history["train_loss"], label="Swin UNETR", color="red", linewidth=2)
axes[0].set_title("Training Loss Comparison", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("DiceCE Loss")
axes[0].grid(True, linestyle="--", alpha=0.6)
axes[0].legend()

# Plot Dice
axes[1].plot(unet_history["val_dice"], label=f"3D U-Net (Best: {best_unet_dice:.4f})", color="blue", marker="o")
axes[1].plot(swin_history["val_dice"], label=f"Swin UNETR (Best: {best_swin_dice:.4f})", color="red", marker="s")
axes[1].set_title("Validation Dice Score Comparison", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Evaluation Step")
axes[1].set_ylabel("Dice Score")
axes[1].grid(True, linestyle="--", alpha=0.6)
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Trực quan hóa Lát cắt Overlay giữa hai Mô hình


In [ ]:
# Select 1 validation sample to visualize
val_sample = val_dataset[0]
img_tensor = val_sample["image"].unsqueeze(0).to(device)
gt_mask = val_sample["label"].squeeze().numpy()

# Run inference
unet_model.eval()
swin_model.eval()
with torch.no_grad():
    u_out = sliding_window_inference(img_tensor, roi_size=(128, 128, 128), sw_batch_size=2, predictor=unet_model)
    s_out = sliding_window_inference(img_tensor, roi_size=(128, 128, 128), sw_batch_size=2, predictor=swin_model)
    
    u_pred = torch.argmax(torch.softmax(u_out, dim=1), dim=1).squeeze().cpu().numpy()
    s_pred = torch.argmax(torch.softmax(s_out, dim=1), dim=1).squeeze().cpu().numpy()

mri_slice = val_sample["image"][0].numpy()[:, :, 80]  # Axial middle slice
gt_slice = gt_mask[:, :, 80]
u_slice = u_pred[:, :, 80]
s_slice = s_pred[:, :, 80]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(mri_slice, cmap="gray")
axes[0].set_title("MRI T1n Input", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(mri_slice, cmap="gray")
axes[1].imshow(np.ma.masked_where(gt_slice == 0, gt_slice), cmap="jet", alpha=0.6)
axes[1].set_title("Ground Truth Mask", fontweight="bold")
axes[1].axis("off")

axes[2].imshow(mri_slice, cmap="gray")
axes[2].imshow(np.ma.masked_where(u_slice == 0, u_slice), cmap="jet", alpha=0.6)
axes[2].set_title("3D U-Net Prediction", fontweight="bold")
axes[2].axis("off")

axes[3].imshow(mri_slice, cmap="gray")
axes[3].imshow(np.ma.masked_where(s_slice == 0, s_slice), cmap="jet", alpha=0.6)
axes[3].set_title("Swin UNETR Prediction", fontweight="bold")
axes[3].axis("off")

plt.tight_layout()
plt.show()
print("🎉 Hoàn thành Huấn luyện và Trực quan hóa trong Notebook!")